# Reprodução do modelo isobárico: $B^{\pm}\to\pi^{\pm}\pi^+\pi^-$

Este notebook é o benchmark signal-only da análise LHCb publicada em [Phys. Rev. D 101, 012006](https://cds.cern.ch/record/2689374/files/1909.05212.pdf).

O objetivo inicial é avaliar os coeficientes cartesianos publicados e calcular as fit fractions por carga e as assimetrias $A_{CP}$. A reprodução estatística do ajuste experimental exige os eventos de dados da análise; este notebook valida a álgebra de amplitudes e a normalização do fitter.

## Convenções

Para cada componente, usamos a convenção da Tabela XXI:

$$c_q=(x+q\,\delta x)+i(y+q\,\delta y),\qquad q=+1\;(B^+),\;q=-1\;(B^-).$$

A contribuição $\rho(770)^0-\omega(782)$ é agrupada incluindo o termo de interferência entre os dois componentes.

Orientação angular: `pair=(2, 0)` usa a partícula 3 para definir o ângulo com a bachelor 2, como `LauKinematics::calcHelicities` para o par 1,3. A simetrização gera o par 3,2 com a mesma orientação. Inverter essa ordem muda o sinal das ondas ímpares relativamente às pares.

In [ ]:
import sys
from pathlib import Path

# Make the notebook work from the repository or from notebooks/benchmark.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "dalitzplotfitter").is_dir():
        sys.path.insert(0, str(candidate / "src"))
        break

import numpy as np
import pandas as pd

from dalitzplotfitter import (
    DecayChannel, DecayModel, PipiKKRescattering,
    RealImag, RelativisticBreitWigner, Resonance, RhoOmegaMixing,
    SigmaPole, enable_x64, ZemachP
)

enable_x64()

In [ ]:
# O notebook é autocontido: não depende de um módulo Python externo.
# Massas/larguras em GeV. A fonte dos números está documentada na célula seguinte.
RESCATTERING_CONVENTION = "laura"  # "paper" reproduz a expressão impressa
PAPER_MASS_WIDTHS = {
    "rho770": (0.7708, 0.1534),       # ajuste isobárico, Tabela XX
    "omega782": (0.78265, 0.00849),   # Laura++/PDG
    "f2_1270": (1.256, 0.1867),      # ajuste isobárico,
    "rho1450": (1.465, 0.400),        # Laura++/PDG
    "rho3_1690": (1.686, 0.186),      # Laura++/PDG
    "sigma": (0.563, 0.350),          # polo ajustado, Tabela XXII
}
COEFFICIENTS = {
    "rho770": (1.000, 0.000, -0.003, 0.000),
    "omega782": (0.091, -0.007, 0.000, -0.022),
    "f2_1270": (0.291, 0.204, -0.002, -0.179),
    "rho1450": (-0.223, 0.191, 0.031, 0.068),
    "rho3_1690": (0.073, -0.045, 0.044, -0.013),
    "rescattering": (0.142, -0.040, -0.047, -0.027),
    "sigma": (-0.485, 0.284, 0.231, 0.270),
}
PUBLISHED_FRACTIONS = {
    "+": {"rho770_omega782": 57.9, "f2_1270": 5.1, "rho1450": 6.2, "rho3_1690": 1.0, "rescattering": 0.8, "sigma": 22.2},
    "-": {"rho770_omega782": 53.3, "f2_1270": 12.6, "rho1450": 4.3, "rho3_1690": 0.1, "rescattering": 2.0, "sigma": 27.9},
}
PUBLISHED_ACP = {"rho770": 0.7, "omega782": -4.8, "f2_1270": 46.8, "rho1450": -12.9, "rho3_1690": -80.1, "rescattering": 44.7, "sigma": 16.0}

def coefficient(name, charge):
    x, y, dx, dy = COEFFICIENTS[name]
    return complex(x + charge * dx, y + charge * dy)

def components(charge):
    c = {name: coefficient(name, charge) for name in COEFFICIENTS}
    rho_mass, rho_width = PAPER_MASS_WIDTHS["rho770"]
    omega_mass, omega_width = PAPER_MASS_WIDTHS["omega782"]
    f2_mass, f2_width = PAPER_MASS_WIDTHS["f2_1270"]
    rho1450_mass, rho1450_width = PAPER_MASS_WIDTHS["rho1450"]
    rho3_mass, rho3_width = PAPER_MASS_WIDTHS["rho3_1690"]
    sigma_mass, sigma_width = PAPER_MASS_WIDTHS["sigma"]
    return [
        # Os dois termos usam os mesmos fatores P-wave do rho; o termo omega
        # mantém sua própria massa/largura apenas dentro de R_omega(m).
        Resonance("rho770", (2, 0), RealImag(c["rho770"].real, c["rho770"].imag),
                  lineshape=RhoOmegaMixing(component="rho", rho_mass=rho_mass,
                                           rho_width=rho_width, omega_mass=omega_mass,
                                           omega_width=omega_width),
                  mass=rho_mass, width=rho_width, spin=1, angular=ZemachP(),
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("omega782", (2, 0), RealImag(c["omega782"].real, c["omega782"].imag),
                  lineshape=RhoOmegaMixing(component="omega", rho_mass=rho_mass,
                                           rho_width=rho_width, omega_mass=omega_mass,
                                           omega_width=omega_width),
                  mass=rho_mass, width=rho_width, spin=1, angular=ZemachP(),
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("f2_1270", (2, 0), RealImag(c["f2_1270"].real, c["f2_1270"].imag),
                  lineshape=RelativisticBreitWigner(), mass=f2_mass, width=f2_width,
                  spin=2, angular=ZemachP(), resonance_radius=4.0, parent_radius=4.0),
        Resonance("rho1450", (2, 0), RealImag(c["rho1450"].real, c["rho1450"].imag),
                  lineshape=RelativisticBreitWigner(), mass=rho1450_mass, width=rho1450_width,
                  spin=1, angular=ZemachP(), resonance_radius=4.0, parent_radius=4.0),
        Resonance("rho3_1690", (2, 0), RealImag(c["rho3_1690"].real, c["rho3_1690"].imag),
                  lineshape=RelativisticBreitWigner(), mass=rho3_mass, width=rho3_width,
                  spin=3, angular=ZemachP(), resonance_radius=4.0, parent_radius=4.0),
        # Rescattering não tem polo: 1.0--1.5 GeV é a janela da parametrização.
        Resonance("rescattering", (2, 0), RealImag(c["rescattering"].real, c["rescattering"].imag),
                  lineshape=PipiKKRescattering(convention=RESCATTERING_CONVENTION), mass=1.0, width=0.0, spin=0,
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("sigma", (2, 0), RealImag(c["sigma"].real, c["sigma"].imag),
                  lineshape=SigmaPole(), mass=sigma_mass, width=sigma_width, spin=0,
                  angular=ZemachP(), resonance_radius=4.0, parent_radius=4.0),
    ]

def make_model(charge, resolution):
    pion = "pi+" if charge > 0 else "pi-"
    other = "pi-" if charge > 0 else "pi+"
    return DecayModel(DecayChannel("B+" if charge > 0 else "B-", (pion, pion, other)), components(charge), normalization_method="gauss-legendre", normalization_resolution=resolution, normalization_order_m13=resolution, normalization_order_m23=resolution, normalize_components=True)

## Massas e larguras usadas

A massa e a largura do $\rho(770)^0$ são os valores ajustados no isobárico (Tabela XX), e os parâmetros do polo $\sigma$ vêm da Tabela XXII. Para $\omega(782)$, $f_2(1270)$, $\rho(1450)^0$ e $\rho_3(1690)^0$, a Tabela III especifica as linhashapes mas não imprime os números: usamos os valores nominais do catálogo Laura++/PDG. O rescattering não recebe massa/largura de ressonância; sua parametrização é definida na janela $1{,}0<m<1{,}5$ GeV.

In [ ]:
mass_widths = pd.DataFrame.from_dict(
    PAPER_MASS_WIDTHS, orient="index", columns=["mass [GeV]", "width [GeV]"]
)
mass_widths.index.name = "component"
mass_widths

## Convenção do rescattering e diferenças restantes

A opção `RESCATTERING_CONVENTION="laura"` usa a fase global $i$ e o fator de produção $[(1+s/\lambda_{\pi\pi}^2)(1+s/\lambda_{KK}^2)]^{-1}$, $s=m^2$, do `LauRescatteringRes.cc` local (Laura++ 3.8). A opção `"paper"` mantém a expressão impressa. Ambas mantêm aqui a janela 1.0–1.5 GeV; o Laura++ original começa no limiar KK.

Essa escolha melhora as interferências, mas ainda não certifica a reprodução da configuração original (Laura++ 3.5). Com a convenção Laura e ordem 2000, a fração rho-omega é 59.8915% (B+) e 54.0558% (B-), versus 57.9% e 53.3%. Na ordem 1400, é 59.8934% e 54.0592%. A diferença de integração nesse teste é muito menor que a discrepância com o paper.

O suplemento identifica `Fit3piIsobar.cc` como a configuração completa. Esse arquivo ainda não foi recuperado: o CDS retornou uma página de verificação automática e o APS negou acesso. Os fontes arXiv consultados continham matrizes de correlação, mas não esse arquivo. A normalização/mapeamento dos coeficientes da mistura permanece pendente; não reajustamos coeficientes para forçar as frações publicadas.


## Coeficientes cartesianos publicados

Os valores abaixo são os centrais da Tabela XXI. Os componentes de erro não são usados nesta primeira etapa.

In [ ]:
coefficients = pd.DataFrame(
    COEFFICIENTS, index=["x", "y", "dx", "dy"]
).T
coefficients

## Construção dos modelos $B^+$ e $B^-$

`resolution` controla explicitamente as ordens Gauss–Legendre em m13 e m23. Apenas mudar `normalization_resolution` não alterava essa malha: esse parâmetro controla Square Dalitz. Compare ordens 1000, 1400 e 2000 para estudar convergência.


In [ ]:
resolution = 1000
plus_model = make_model(+1, resolution)
minus_model = make_model(-1, resolution)

print("B+ components:", [c.name for c in plus_model.amplitude_model.components])
print("B- components:", [c.name for c in minus_model.amplitude_model.components])

In [ ]:
def fractions_and_interference(model):
    cache = model._fraction_cache(None, None)
    fractions = np.asarray(cache.fit_fractions({})) * 100.0
    interference = np.asarray(cache.interference_fractions({})) * 100.0
    names = [component.name for component in cache.components]
    result = dict(zip(names, fractions, strict=True))
    result["rho770_omega782"] = (
        result["rho770"] + result["omega782"] + interference[0, 1]
    )
    return result, interference

plus_fractions, plus_interference = fractions_and_interference(plus_model)
minus_fractions, minus_interference = fractions_and_interference(minus_model)

## Fit fractions por carga

As Tabelas 12 e 13 apresentam o valor diagonal de cada componente. A soma das frações não precisa ser 100%, porque os termos de interferência não estão incluídos nas frações diagonais.

In [ ]:
fraction_names = [
    "rho770_omega782", "f2_1270", "rho1450",
    "rho3_1690", "rescattering", "sigma",
]
fractions = pd.DataFrame({
    "B+ evaluator [%]": [plus_fractions[n] for n in fraction_names],
    "B+ paper [%]": [PUBLISHED_FRACTIONS["+"][n] for n in fraction_names],
    "B- evaluator [%]": [minus_fractions[n] for n in fraction_names],
    "B- paper [%]": [PUBLISHED_FRACTIONS["-"][n] for n in fraction_names],
}, index=fraction_names)
fractions["delta B+ [%]"] = fractions["B+ evaluator [%]"] - fractions["B+ paper [%]"]
fractions["delta B- [%]"] = fractions["B- evaluator [%]"] - fractions["B- paper [%]"]
fractions.round(4)

## Frações de interferência

A matriz abaixo contém os termos $2\operatorname{Re}(A_iA_j^*)$ normalizados pela intensidade total. Ela é necessária para verificar a contribuição agrupada $\rho-\omega$ e a soma total do modelo.

In [ ]:
component_names = [c.name for c in plus_model.amplitude_model.components]
plus_interference_table = pd.DataFrame(plus_interference, index=component_names, columns=component_names)
minus_interference_table = pd.DataFrame(minus_interference, index=component_names, columns=component_names)

print("B+ interference fractions [%]")
display(plus_interference_table.round(4))
print("B- interference fractions [%]")
display(minus_interference_table.round(4))

## Assimetria $A_{CP}$

Usamos os coeficientes da Tabela XXI e a Eq. (33):

$$A_{CP}^j = \frac{|c_j^-|^2-|c_j^+|^2}{|c_j^-|^2+|c_j^+|^2}.$$

A assimetria entre fit fractions tem denominadores totais diferentes e não representa este observável. Os coeficientes impressos têm três casas decimais, o que limita a reprodução dos valores centrais, especialmente para omega.


In [ ]:
acp_names = list(PUBLISHED_ACP)
acp = pd.DataFrame({
    "evaluator [%]": [
        100.0 * (abs(coefficient(n, -1))**2 - abs(coefficient(n, +1))**2) /
        (abs(coefficient(n, -1))**2 + abs(coefficient(n, +1))**2)
        for n in acp_names
    ],
    "paper [%]": [PUBLISHED_ACP[n] for n in acp_names],
}, index=acp_names)
acp["delta [%]"] = acp["evaluator [%]"] - acp["paper [%]"]
acp.round(4)

## Próxima etapa: toy closure

Depois que as diferenças acima forem explicadas pelas convenções de linha de forma, o próximo bloco deve gerar toys de sinal para as duas cargas e ajustar os coeficientes. O teste de certificação será:

1. injetar os coeficientes da Tabela XXI;
2. gerar amostras $B^+$ e $B^-$;
3. ajustar os coeficientes;
4. verificar recuperação dentro das incertezas estatísticas;
5. recalcular fit fractions e $A_{CP}$.

Isso valida o fitter com sinal puro, mas não substitui a reprodução do ajuste experimental com eficiência e background.